# Events - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, timestamp_millis

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.retailrocket_events"
target_table = f"{catalog}.silver.retailrocket_events"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- timestamp: long (nullable = true)
 |-- visitorid: string (nullable = true)
 |-- event: string (nullable = true)
 |-- itemid: string (nullable = true)
 |-- transactionid: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

timestamp,visitorid,event,itemid,transactionid,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1432701181734,133468,view,437804,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432700141070,624436,view,212955,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432700032079,251018,view,277505,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432702843159,877542,view,40593,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432701735130,225705,view,219512,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432700769293,592878,view,324963,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432700604598,1258752,view,335975,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432700175559,914184,view,437065,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432703684910,1203785,view,184460,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432703394056,580305,view,176946,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 2756101
Number of columns: 12


In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


In [0]:
for column, dtype in bronze_df.dtypes[:5]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

timestamp
Null count: 0
Distinct count: 2750455
--------------------
visitorid
Null count: 0
Distinct count: 1407580
Extra whitespace row count: 0
--------------------
event
Null count: 0
Distinct count: 3
Extra whitespace row count: 0
--------------------
itemid
Null count: 0
Distinct count: 235061
Extra whitespace row count: 0
--------------------
transactionid
Null count: 2733644
Distinct count: 17672
Extra whitespace row count: 0
--------------------


In [0]:
display(bronze_df.groupBy("event").count())

event,count
addtocart,69332
view,2664312
transaction,22457


In [0]:
print(bronze_df.filter(col("event").isin(["view", "addtocart"]) & col("transactionid").isNotNull()).count())

0


In [0]:
distinct_event_rows = (
    bronze_df
    .select(["timestamp", "visitorid", "event", "itemid", "transactionid"])
    .distinct()
    .count()
)
print("Distinct row count:", distinct_event_rows)

print("Duplicate excess rows:", bronze_df.count() - distinct_event_rows)

Distinct row count: 2755641
Duplicate excess rows: 460


There is no source provided primary key for this table.

## Transform to Silver

In [0]:
silver_df = bronze_df.dropDuplicates(["timestamp", "visitorid", "event", "itemid", "transactionid"])

In [0]:
silver_df = (
    silver_df
    .withColumnRenamed("visitorid", "visitor_id")
    .withColumnRenamed("event", "event_type")
    .withColumnRenamed("itemid", "item_id")
    .withColumnRenamed("transactionid", "transaction_id")
)

In [0]:
silver_df = silver_df.withColumn(
    "event_timestamp",
    timestamp_millis("timestamp")
)

In [0]:
silver_df = silver_df.withColumnRenamed(
    "timestamp",
    "event_timestamp_millis"
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- event_timestamp_millis: long (nullable = true)
 |-- visitor_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)



In [0]:
display(silver_table_df.limit(10))

event_timestamp_millis,visitor_id,event_type,item_id,transaction_id,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset,event_timestamp
1433221923240,810725,view,443030,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-02T05:12:03.240Z
1433223697356,598426,view,156489,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-02T05:41:37.356Z
1433223176926,629333,view,128394,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-02T05:32:56.926Z
1433222571026,293285,view,288868,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-02T05:22:51.026Z
1433222646781,529601,view,161166,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-02T05:24:06.781Z
1433222012357,266709,view,190545,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-02T05:13:32.357Z
1433194792189,24769,view,55364,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-01T21:39:52.189Z
1433194453729,610183,view,396238,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-01T21:34:13.729Z
1433191990038,481837,view,247307,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-01T20:53:10.038Z
1433219179174,304146,view,42002,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events,2015-06-02T04:26:19.174Z


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 2756101
Silver row count: 2755641
